**Reference: https://python.langchain.com/v0.1/docs/use_cases/sql/large_db/**

What happens in this notebook:

### **Table Model Definition**
   - **`Table` Class**: This is a simple Pydantic model representing a SQL table. It has one attribute, `name`, which is a string and is described as "Name of table in SQL database."
     - This model is used in the extraction process to match relevant SQL tables based on the user's query.

### **Helper Function - `get_tables`**
   - **`get_tables`**: This function takes a list of `Table` objects (i.e., categories such as "Music" or "Business") and returns a list of corresponding SQL table names based on the category.
     - For example, if the category is `"Music"`, the tables `"Album"`, `"Artist"`, `"Genre"`, etc., are added to the result.
     - Similarly, for `"Business"`, the corresponding tables like `"Customer"`, `"Employee"`, etc., are included.

### **Designing the agent for the large DB**

- **Step 1: Initialize LLM (`sql_agent_llm`)**: The LLM is instantiated with a given model (e.g., `"gpt-3.5-turbo"`) and temperature. The temperature controls how creative/random the model's responses are.
- **Step 2: Connect to the SQL Database (`db`)**: The connection to the Chinook SQLite database is established. The database URI is constructed using the `sqldb_directory` provided.
- **Step 3: Define Category Chain (`category_chain`)**: The `category_chain_system` is defined, which is a string explaining the categories available (like "Music" and "Business"). This chain determines which SQL tables are relevant to the user query based on the category.
- **Step 4: Chain Creation**:
- **`category_chain`**: This uses the `create_extraction_chain_pydantic` function, which creates an extraction chain that identifies relevant SQL tables from the user's question using the `Table` Pydantic model and the LLM.
- **`table_chain`**: A chain is formed by combining the output from `category_chain` with the `get_tables` function, so it maps categories to the actual SQL tables.
- **Step 5: Query Chain (`query_chain`)**: This creates a SQL query chain using the LLM and the database (`self.db`). It takes the SQL tables and constructs a query.
- **Step 6: Table Chain Input Handling**: The `"question"` key from the user input is mapped to the `"input"` key expected by the `table_chain`. This enables the chain to process user queries correctly.
- **Step 7: Full Chain Construction**: Finally, the full chain (`full_chain`) is created by combining:
1. **`RunnablePassthrough.assign`**: This sets up a step that assigns the `table_names_to_use` using the result of the `table_chain`.
2. **`query_chain`**: Executes the SQL query once the relevant tables are identified.

In [ ]:
# %pip install langchain-chroma
# %pip install pyprojroot
# %pip install langchain_huggingface
# %pip install sentence-transformers
# %pip install langchain
# %pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
from dotenv import load_dotenv
from pyprojroot import here
from typing import List
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq
from pprint import pprint
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
load_dotenv()


True

**Set the environment variables and load the LLM**

In [12]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

sql_agent_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
table_extractor_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
# llm = ChatOpenAI(model="gpt-4o")

In [13]:
sqldb_directory = here("data/Chinook.db")
db = SQLDatabase.from_uri(f"sqlite:///{sqldb_directory}")
print(db.dialect)
print(db.get_usable_table_names())
db.run("SELECT * FROM Artist LIMIT 10;")

sqlite
['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

**Prepare the `Table` class**

In [14]:
class Table(BaseModel):
    """
    Represents a table in the SQL database.

    Attributes:
        name (str): The name of the table in the SQL database.
    """
    name: str = Field(description="Name of table in SQL database.")

### **Strategy A:**

In [15]:
table_names = "\n".join(db.get_usable_table_names())
pprint(table_names)

('Album\n'
 'Artist\n'
 'Customer\n'
 'Employee\n'
 'Genre\n'
 'Invoice\n'
 'InvoiceLine\n'
 'MediaType\n'
 'Playlist\n'
 'PlaylistTrack\n'
 'Track')


In [17]:
load_dotenv()
class Table(BaseModel):
    table_names: list[str] = Field(
        description="List of relevant SQL table names"
    )

system = f"""Return the names of ALL the SQL tables that MIGHT be relevant to the user question.
The tables are:

{table_names}

Remember to include ALL POTENTIALLY RELEVANT tables, even if you're not sure that they're needed."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{input}")]
)
table_chain = prompt | table_extractor_llm.with_structured_output(Table)

result = table_chain.invoke(
    {"input": "What are all the genres of Alanis Morisette songs"}
)

print(result)

table_names=['Artist', 'Track', 'Genre']


### **Strategy B:**

In [38]:
class Table(BaseModel):
    table_names: list[str] = Field(
        description="List of selected SQL table names"
    )
system = f"""You will receive a question.

If the question is about **Music**, return **ALL** these tables:
  - "Album"
  - "Artist"
  - "Genre"
  - "MediaType"
  - "Playlist"
  - "PlaylistTrack"
  - "Track"

If the question is about **Business**, return **ALL** these tables:
  - "Customer"
  - "Employee"
  - "Invoice"
  - "InvoiceLine"

If you are unsure, return the full list of all available tables for both Music and Business categories."""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{input}")]
)
table_chain = prompt | table_extractor_llm.with_structured_output(Table)
result = table_chain.invoke(
    {"input": "What are all the genres of Alanis Morisette songs"}
)
print(result.table_names)

['Album', 'Artist', 'Genre', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


### **Strategy C:**

- **Step 1: Define the category**

In [39]:
class Table(BaseModel):
    category: list[str] = Field(
        description="List of relevant table categories (e.g. Music, Business)"
    )
system = """Return the names of the SQL tables that are relevant to the user question. \
The tables are:
Music
Business"""
prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{input}")]
)
category_chain = prompt | table_extractor_llm.with_structured_output(Table)

In [40]:
category_chain.invoke({"input": "What are all the genres of Alanis Morisette songs"})

Table(category=['Music'])

- **Step 2: Execute the python function**

In [41]:
load_dotenv()
class Category(BaseModel):
    name: str = Field(
        description="Category name, either 'Music' or 'Business'"
    )
class CategoryResponse(BaseModel):
    categories: List[Category] = Field(
        description="List of relevant table categories"
    )
system = """Return the names of the SQL tables that are relevant to the user question.
The tables are:

Music
Business"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system), ("human", "{input}")]
)
category_chain = prompt | table_extractor_llm.with_structured_output(
    CategoryResponse
)
def get_tables(response: CategoryResponse) -> List[str]:
    """Maps Category response to corresponding SQL table names."""
    tables = []
    for category in response.categories:
        if category.name == "Music":
            tables.extend(
                [
                    "Album",
                    "Artist",
                    "Genre",
                    "MediaType",
                    "Playlist",
                    "PlaylistTrack",
                    "Track",
                ]
            )
        elif category.name == "Business":
            tables.extend(["Customer", "Employee", "Invoice", "InvoiceLine"])
    return tables

table_chain = category_chain | get_tables
result = table_chain.invoke(
    {"input": "What are all the genres of Alanis Morisette songs"}
)
print(result)

['Album', 'Artist', 'Genre', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [23]:
import sys
print(sys.version)
print(sys.executable)

3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:12:32) [Clang 20.1.8 ]
/opt/miniconda3/envs/a1/bin/python


In [ ]:
# %pip install -U langchain-classic langchain-community langchain-groq

Note: you may need to restart the kernel to use updated packages.


### **Final step:**

**Attach the desired strategy to your SQL agent**

In [48]:
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase
table_chain_for_full_chain = {"input": itemgetter("question")} | table_chain
query_chain = create_sql_query_chain(sql_agent_llm,db)
full_chain = (RunnablePassthrough.assign(table_names_to_use=table_chain_for_full_chain)| query_chain)

**Test the agent**

In [50]:
query = full_chain.invoke(
    {"question": "What are all the genres of Alanis Morisette songs"}
)
print(query)

SQLQuery: SELECT DISTINCT g."Name" FROM "Genre" g JOIN "Track" t ON g."GenreId" = t."GenreId" JOIN "Album" a ON t."AlbumId" = a."AlbumId" JOIN "Artist" ar ON a."ArtistId" = ar."ArtistId" WHERE ar."Name" = 'Alanis Morisette' LIMIT 5


**Prepare the tool (Don't run the following cell)**

In [56]:
import sys
from pyprojroot import here
root_dir = str(here())
if root_dir not in sys.path:
    sys.path.append(root_dir)
    
from src.agent_graph.load_tools_config import LoadToolsConfig

TOOLS_CFG = LoadToolsConfig()

class ChinookSQLAgent:
    """
    A specialized SQL agent that interacts with the Chinook SQL database using an LLM (Large Language Model).

    The agent handles SQL queries by mapping user questions to relevant SQL tables based on categories like "Music"
    and "Business". It uses an extraction chain to determine relevant tables based on the question and then
    executes queries against the database using the appropriate tables.

    Attributes:
        sql_agent_llm (ChatOpenAI): The language model used for interpreting and interacting with the database.
        db (SQLDatabase): The SQL database object, representing the Chinook database.
        full_chain (Runnable): A chain of operations that maps user questions to SQL tables and executes queries.

    Methods:
        __init__: Initializes the agent by setting up the LLM, connecting to the SQL database, and creating query chains.

    Args:
        sqldb_directory (str): The directory where the Chinook SQLite database file is located.
        llm (str): The name of the LLM model to use (e.g., "gpt-3.5-turbo").
        llm_temperature (float): The temperature setting for the LLM, controlling the randomness of responses.
    """

    def __init__(self, sqldb_directory: str, llm: str, llm_temperature: float) -> None:
        """Initializes the ChinookSQLAgent with the LLM and database connection.

        Args:
            sqldb_directory (str): The directory path to the SQLite database file.
            llm (str): The LLM model identifier (e.g., "gpt-3.5-turbo").
            llm_temerature (float): The temperature value for the LLM, determining the randomness of the model's output.
        """
        self.sql_agent_llm = ChatGroq(model=llm, temperature=llm_temperature)

        self.db = SQLDatabase.from_uri(f"sqlite:///{sqldb_directory}")
        print(self.db.get_usable_table_names())
        category_chain_system = """Return the names of the SQL tables that are relevant to the user question. \
        The tables are:

        Music
        Business"""
        
        category_prompt = ChatPromptTemplate.from_messages([("system", category_chain_system),("human", "{input}")])

        category_chain = (category_prompt| self.sql_agent_llm.with_structured_output(Table))

        query_chain = create_sql_query_chain(self.sql_agent_llm, self.db)
        # Convert "question" key to the "input" key expected by current table_chain.
        table_chain = {"input": itemgetter("question")} | table_chain
        # Set table_names_to_use using table_chain.
        self.full_chain = RunnablePassthrough.assign(
            table_names_to_use=table_chain) | query_chain


@tool
def query_chinook_sqldb(query: str) -> str:
    """Query the Chinook SQL Database. Input should be a search query."""
    # Create an instance of ChinookSQLAgent
    agent = ChinookSQLAgent(
        sqldb_directory=TOOLS_CFG.chinook_sqldb_directory,
        llm=TOOLS_CFG.chinook_sqlagent_llm,
        llm_temerature=TOOLS_CFG.chinook_sqlagent_llm_temperature
    )

    query = agent.full_chain.invoke({"question": query})

    return agent.db.run(query)

In [57]:
result = category_chain.invoke({
    "input": "What are all the genres of Alanis Morissette songs?"
})

print(result)

categories=[Category(name='Music')]


In [58]:
result = table_chain.invoke(
    {"input": "What are all the genres of Alanis Morisette songs"}
)

print(result)

['Album', 'Artist', 'Genre', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
